# Tracing for Different Types of Runs

### Types of Runs

LangSmith supports many different types of Runs - you can specify what type your Run is in the @traceable decorator. The types of runs are:

- LLM: Invokes an LLM
- Retriever: Retrieves documents from databases or other sources
- Tool: Executes actions with function calls
- Chain: Default type; combines multiple Runs into a larger process
- Prompt: Hydrates a prompt to be used with an LLM
- Parser: Extracts structured data

### Setup

In [1]:
# You can set them inline!
import os
os.environ["GOOGLE_API_KEY"] = ""
os.environ["LANGSMITH_API_KEY"] = ""
os.environ["LANGSMITH_TRACING"] = "true"


In [2]:
# Or you can use a .env file
from dotenv import load_dotenv
load_dotenv(dotenv_path=".env", override=True)


True

### LLM Runs for Chat Models

LangSmith provides special rendering and processing for LLM traces. In order to make the most of this feature, you must log your LLM traces in a specific format.

For chat-style models, inputs should be a list of message dictionaries with `role` and `content` keys. This matches the format used by Gemini and other chat-based LLMs.

A Gemini response is commonly returned as a `GenerateContentResponse` object, and the text is typically read from `.text`. When tracing through LangSmith, you can also attach metadata such as:
- ls_provider: The provider of the model, e.g. "google"
- ls_model_name: The name of the model, e.g. "gemini-3.6-flash"

In [3]:
from langsmith import traceable

inputs = [
  {"role": "system", "content": "You are a helpful assistant."},
  {"role": "user", "content": "I'd like to book a table for two."},
]

output = {
  "choices": [
      {
          "message": {
              "role": "assistant",
              "content": "Sure, what time would you like to book the table for?"
          }
      }
  ]
}


# Can also use one of:
# output = {
#     "message": {
#         "role": "assistant",
#         "content": "Sure, what time would you like to book the table for?"
#     }
# }
#
# output = {
#     "role": "assistant",
#     "content": "Sure, what time would you like to book the table for?"
# }
#
# output = ["assistant", "Sure, what time would you like to book the table for?"]

@traceable(
  # TODO: Add an run_type="llm"
    run_type="llm"
)
def chat_model(messages: list):
  return output

chat_model(inputs)

{'choices': [{'message': {'role': 'assistant',
    'content': 'Sure, what time would you like to book the table for?'}}]}

### Retriever Runs + Documents

Many LLM applications require looking up documents from vector databases, knowledge graphs, or other types of indexes. Retriever traces are a way to log the documents that are retrieved by the retriever. LangSmith provides special rendering for retrieval steps in traces to make it easier to understand and diagnose retrieval issues. In order for retrieval steps to be rendered correctly, a few small steps need to be taken.

1. Annotate the retriever step with run_type="retriever".
2. Return a list of Python dictionaries or TypeScript objects from the retriever step. Each dictionary should contain the following keys:
    - page_content: The text of the document.
    - type: This should always be "Document".
    - metadata: A python dictionary or TypeScript object containing metadata about the document. This metadata will be displayed in the trace.

In [4]:
from langsmith import traceable

def _convert_docs(results):
  return [
      {
          "page_content": r,
          "type": "Document",
          "metadata": {"foo": "bar"}
      }
      for r in results
  ]

@traceable(
    # TODO: Add an run_type="retriever"
    run_type="retriever"
)
def retrieve_langsmith_docs(query):
  # Retriever returning hardcoded dummy documents.
  # In production, this could be a real vector datatabase or other document index.
  contents = ["LangSmith Document contents 1", "LangSmith Document contents 2", "LangSmith Document contents 3"]
  return _convert_docs(contents)

retrieve_langsmith_docs("User query")

[{'page_content': 'LangSmith Document contents 1',
  'type': 'Document',
  'metadata': {'foo': 'bar'}},
 {'page_content': 'LangSmith Document contents 2',
  'type': 'Document',
  'metadata': {'foo': 'bar'}},
 {'page_content': 'LangSmith Document contents 3',
  'type': 'Document',
  'metadata': {'foo': 'bar'}}]

### Tool Calling

LangSmith can still render tool-style interactions in traces, even when using Gemini via the Google GenAI SDK. The important part is to log consistent structured messages and tool metadata, including function names, arguments, and the tool response payload.

In [5]:
from langsmith import traceable
import json
import os
from google import genai

client = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))
chat = client.chats.create(model="gemini-3.6-flash")

@traceable(run_type="tool")
def search_web(query: str):
    return f"No web result found for query: {query}"

@traceable(run_type="llm")
def call_gemini(messages: list) -> str:
    prompt = "\n".join(f"{m['role']}: {m['content']}" for m in messages)
    response = chat.send_message(prompt)
    return response.text

@traceable(run_type="chain")
def respond(inputs):
    response_text = call_gemini(inputs)
    query = "weather in New York City"
    tool_result = {
        "role": "tool",
        "content": json.dumps({
            "query": query,
            "results": search_web(query),
        })
    }
    follow_up = inputs + [
        {"role": "assistant", "content": response_text},
        tool_result,
    ]
    return call_gemini(follow_up)

inputs = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "What is the weather today in New York City?"},
]

respond(inputs)


"I tried searching for the latest updates, but I'm still unable to retrieve real-time weather results right now. Please check a weather app or website like Weather.com, AccuWeather, or Apple/Google Weather for today's forecast in New York City."